# MLP from Scratch (NumPy)
Implements a simple Multi-Layer Perceptron using only NumPy for forward pass, backpropagation, weight updates, and training on MNIST.

In [1]:
import numpy as np
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(-1,784).astype(np.float32)/255.0
X_test = X_test.reshape(-1,784).astype(np.float32)/255.0

def one_hot(y, num_classes=10):
    return np.eye(num_classes)[y]

Y_train = one_hot(y_train)
Y_test = one_hot(y_test)

print(X_train.shape, Y_train.shape)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
(60000, 784) (60000, 10)


In [2]:
def relu(x):
    return np.maximum(0,x)

def relu_derivative(x):
    return (x>0).astype(float)

def softmax(z):
    z = z - np.max(z,axis=1,keepdims=True)
    e = np.exp(z)
    return e/np.sum(e,axis=1,keepdims=True)

def cross_entropy(y_true,y_pred):
    eps = 1e-12
    y_pred = np.clip(y_pred,eps,1-eps)
    return -np.mean(np.sum(y_true*np.log(y_pred),axis=1))


In [3]:
class MLP:
    def __init__(self,input_size=784,h1=128,h2=64,output_size=10,lr=0.01):
        self.lr=lr
        self.W1=np.random.randn(input_size,h1)*np.sqrt(2/input_size)
        self.b1=np.zeros((1,h1))
        self.W2=np.random.randn(h1,h2)*np.sqrt(2/h1)
        self.b2=np.zeros((1,h2))
        self.W3=np.random.randn(h2,output_size)*np.sqrt(2/h2)
        self.b3=np.zeros((1,output_size))

    def forward(self,X):
        self.Z1=X@self.W1+self.b1
        self.A1=relu(self.Z1)
        self.Z2=self.A1@self.W2+self.b2
        self.A2=relu(self.Z2)
        self.Z3=self.A2@self.W3+self.b3
        self.A3=softmax(self.Z3)
        return self.A3

    def backward(self,X,Y):
        m=X.shape[0]
        dZ3=(self.A3-Y)/m
        dW3=self.A2.T@dZ3
        db3=np.sum(dZ3,axis=0,keepdims=True)

        dA2=dZ3@self.W3.T
        dZ2=dA2*relu_derivative(self.Z2)
        dW2=self.A1.T@dZ2
        db2=np.sum(dZ2,axis=0,keepdims=True)

        dA1=dZ2@self.W2.T
        dZ1=dA1*relu_derivative(self.Z1)
        dW1=X.T@dZ1
        db1=np.sum(dZ1,axis=0,keepdims=True)

        self.W3-=self.lr*dW3
        self.b3-=self.lr*db3
        self.W2-=self.lr*dW2
        self.b2-=self.lr*db2
        self.W1-=self.lr*dW1
        self.b1-=self.lr*db1

    def predict(self,X):
        return np.argmax(self.forward(X),axis=1)


In [4]:
epochs=10
batch_size=64

model=MLP(lr=0.01)

for epoch in range(epochs):
    idx=np.random.permutation(len(X_train))
    X_train=X_train[idx]
    Y_train=Y_train[idx]
    y_train=y_train[idx]

    for i in range(0,len(X_train),batch_size):
        Xb=X_train[i:i+batch_size]
        Yb=Y_train[i:i+batch_size]
        out=model.forward(Xb)
        model.backward(Xb,Yb)

    train_pred=model.predict(X_train)
    test_pred=model.predict(X_test)

    train_acc=np.mean(train_pred==y_train)
    test_acc=np.mean(test_pred==y_test)

    loss=cross_entropy(Y_train,model.forward(X_train))

    print(f"Epoch {epoch+1}: Loss={loss:.4f} Train Acc={train_acc:.4f} Test Acc={test_acc:.4f}")


Epoch 1: Loss=0.3998 Train Acc=0.8887 Test Acc=0.8968
Epoch 2: Loss=0.3083 Train Acc=0.9116 Test Acc=0.9163
Epoch 3: Loss=0.2670 Train Acc=0.9244 Test Acc=0.9273
Epoch 4: Loss=0.2423 Train Acc=0.9315 Test Acc=0.9319
Epoch 5: Loss=0.2217 Train Acc=0.9369 Test Acc=0.9368
Epoch 6: Loss=0.2029 Train Acc=0.9417 Test Acc=0.9393
Epoch 7: Loss=0.1883 Train Acc=0.9452 Test Acc=0.9439
Epoch 8: Loss=0.1781 Train Acc=0.9490 Test Acc=0.9459
Epoch 9: Loss=0.1681 Train Acc=0.9522 Test Acc=0.9477
Epoch 10: Loss=0.1551 Train Acc=0.9557 Test Acc=0.9532
